In [31]:
!pip install plotly

In [32]:
!pip install pandas

In [33]:
!pip install nbformat 

In [34]:
!pip install --upgrade "nbformat>=4.2.0"

In [35]:
import pandas as pd
import sqlite3
from sqlite3 import Error
import plotly.graph_objects as go
import numpy as np

In [36]:
def create_connection(path):
    connection = None
    try:
        connection = sqlite3.connect(path)
    except Error as e:
        print(f"The error {e} occured")
    return connection

connection = create_connection("../checking-logs.sqlite")

In [37]:
query = """
SELECT uid, DATE(timestamp) AS timestamp, numTrials AS numTrials
FROM checker
WHERE uid LIKE 'user_%' AND labname = 'project1' AND status = 'ready' 
ORDER BY DATE(timestamp)
"""

df = pd.read_sql(query, connection)
df["timestamp"] = pd.to_datetime(df["timestamp"])
users = df["uid"].unique()
dates = df["timestamp"].unique()

df

,uid,timestamp,numTrials
0,user_4,2020-04-17,1
1,user_4,2020-04-17,2
2,user_4,2020-04-17,3
3,user_4,2020-04-17,4
4,user_4,2020-04-17,5
...,...,...,...
946,user_19,2020-05-15,26
947,user_19,2020-05-15,27
948,user_19,2020-05-15,28
949,user_28,2020-05-15,27


In [38]:
frames = []
for i, date in enumerate(dates):
    curr_data = df[df["timestamp"] <= date]
    frame_data = []
    for user in users:
        user_data = curr_data[curr_data["uid"] == user]
        frame_data.append(go.Scatter(x = user_data["timestamp"], 
                          y = user_data["numTrials"],
                          mode = 'lines+markers',
                          name = user))
    frames.append(go.Frame(data = frame_data))

In [39]:
fig = go.Figure(
    data = frames[0].data, 
    frames=frames,
    layout=go.Layout(
        title = "Dynamic of commits per user in project1",
        xaxis=dict(range=[dates[0], dates[-1]]),
        yaxis=dict(range=[0, df["numTrials"].max() * 1.1]),
        updatemenus=[{"type":"buttons",
                      "buttons":[{
                          "method":"animate",
                          "label":"play", 
                          "args":[None, {"transition": {"duration": 300}}]}]}
        ]
    )
)




fig.show()

In [40]:
connection.close()